### Data Loading

In [1]:
import numpy as np
import struct
import os
import torch
from torch.utils.data import TensorDataset, DataLoader

def load_idx(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(num, rows, cols)

def load_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
        return labels

# Load data
loc_path = os.getcwd()
x_train_full = load_idx(loc_path + '\\Datasets\\train-images.idx3-ubyte') / 255.0
y_train_full = load_labels(loc_path + '\\Datasets\\train-labels.idx1-ubyte')

x_test = load_idx(loc_path + '\\Datasets\\t10k-images.idx3-ubyte') / 255.0
y_test = load_labels(loc_path + '\\Datasets\\t10k-labels.idx1-ubyte')

# Use a subset for quick training
x_train = x_train_full[:1000]
y_train = y_train_full[:1000]

# Convert to PyTorch tensors
x_train_tensor = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)


ModuleNotFoundError: No module named 'torch'

### Neural Network Creation

In [ ]:
import torch
import torch.nn as nn          # Provides neural network building blocks (layers, activations, etc.)
import torch.optim as optim    # Provides optimization algorithms (SGD, Adam, etc.)

class NeuralNetwork(nn.Module):
    def __init__(self): # Initialize the neural network Constructor
        super().__init__()  # Call the constructor of the parent class (nn.Module)
        self.flatten = nn.Flatten()  # Flatten layer: converts input of shape (batch_size, 28, 28) into (batch_size, 784)
        self.linear_relu_stack = nn.Sequential( # This defines a stack of layers that will process the flattened image.
            nn.Linear(28 * 28, 128), # First layer: fully connected layer(a layer of neurons) with 128 neurons
            nn.ReLU(),              # A reliu activation function is applied after the first layer to introduce non-linearity
            nn.Linear(128, 10) # Second layer: fully connected layer with 10 neurons (one for each digit class 0-9)
            # No activation function here because we'll apply softmax later during loss calculation
        )

    def forward(self, x): # Defines the forward pass of the neural network
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()


###  Set up training components

In [ ]:

import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

model.to(device)
loss_fn = nn.CrossEntropyLoss() # Cross-entropy loss for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

Training on: cpu


### Training and Test

In [ ]:
accuracy = 0.0
def train_loop(dataloader, model, loss_fn, optimizer, device):
    model.train() # Set the model to training mode
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device) # Move data to the device (GPU or CPU)
        pred = model(X) # Forward pass: compute predicted outputs by passing inputs to the model
        loss = loss_fn(pred, y) # Compute the loss

        optimizer.zero_grad() # Zero the gradients before running the backward pass
        loss.backward() # Backward pass: compute gradient of the loss with respect to model parameters
        optimizer.step() # Update model parameters

        if batch % 10 == 0:
            print(f"Loss: {loss.item():>7f}")

def test_loop(dataloader, model, loss_fn, device):
    model.eval() # Set the model to evaluation mode
    test_loss, correct = 0.0, 0 
    with torch.no_grad(): # Disable gradient calculation for evaluation
        for X, y in dataloader: # Iterate over the test data
            X, y = X.to(device), y.to(device) # Move data to the gpu or cpu
            logits = model(X) # Forward pass
            test_loss += loss_fn(logits, y).item() # Accumulate the loss
            correct += (logits.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"Test Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}")
    accuracy = 100 * correct

epochs = 5
for epoch in range(epochs):
    print(f"Epoch {epoch+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, optimizer, device)
    test_loop(test_loader, model, loss_fn, device)
print("Training complete!")

Epoch 1
-------------------------------
Loss: 2.304307
Loss: 1.846758
Test Accuracy: 69.9%, Avg loss: 1.572256
Epoch 2
-------------------------------
Loss: 1.500017
Loss: 1.120385
Test Accuracy: 79.1%, Avg loss: 0.976673
Epoch 3
-------------------------------
Loss: 0.938317
Loss: 0.704361
Test Accuracy: 81.0%, Avg loss: 0.716762
Epoch 4
-------------------------------
Loss: 0.668094
Loss: 0.524869
Test Accuracy: 83.2%, Avg loss: 0.588062
Epoch 5
-------------------------------
Loss: 0.470009
Loss: 0.360711
Test Accuracy: 83.8%, Avg loss: 0.550139
Training complete!


### Saving The model

In [14]:
import torch
import numpy as np
import os

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total}")
    print(f"Trainable parameters: {trainable}")
    return total, trainable

def save_model_if_better(model, accuracy, loc_path):
    total, trainable = count_parameters(model)
    model_power = int(accuracy + np.log(trainable))

    models_dir = os.path.join(loc_path, "Models")
    os.makedirs(models_dir, exist_ok=True)

    best_power = -1
    best_model_path = None

    # Check existing models
    for filename in os.listdir(models_dir):
        if filename.startswith("Mnist_Perceptron_Pytorch_") and filename.endswith(".pth"):
            try:
                parts = filename.split("_")
                old_power = int(parts[-2])
                if old_power > best_power:
                    best_power = old_power
                    best_model_path = filename
            except ValueError:
                continue

    # Save if better
    if model_power > best_power:
        save_path = os.path.join(models_dir, f"Mnist_Perceptron_Pytorch_{model_power}_.pth")
        torch.save(model.state_dict(), save_path)
        print(f"✅ New model saved with power {model_power}")
    else:
        print(f"⚠️ Model not saved — existing model has equal or better power ({best_power})")

save_model_if_better(model, accuracy, loc_path)

Total parameters: 101770
Trainable parameters: 101770
✅ New model saved with power 11


### 